# Stream B — fMRI Preprocessing & ROI Extraction

| Field | Value |
|-------|-------|
| **Lead** | Dora Xiang |
| **PI** | Peter Zhou |
| **Dataset** | [ds000030](https://openneuro.org/datasets/ds000030) (UCLA Consortium for Neuropsychiatric Phenomics) |
| **Pipeline** | OpenNeuro → Phenotype Audit → fMRIPrep Derivatives → Nilearn ROI → Feature CSV |
| **Target Output** | `stream_b_features.csv` (per-subject ventral striatum BOLD features) |

> ⚠️ **Resource Note:** For full ds000030 (~272 subjects), run Steps 3-4 on USC CARC or a machine with ≥32 GB RAM. Steps 1-2 can run on Colab.

---
## Cell 1: Install Dependencies

In [ ]:
# ============================================================
# Cell 1: Dependencies
# ============================================================
!pip install -q nilearn nibabel pandas matplotlib seaborn openneuro-py tqdm

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('✅ Dependencies installed')

---
## Cell 2: Mount Google Drive & Configure Paths

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ---- UPDATE THESE PATHS ---- #
DRIVE_BASE = Path('/content/drive/MyDrive/NSG/Stream_B')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Local working directory (Colab's ephemeral disk — faster I/O)
WORK_DIR = Path('/content/ds000030')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Output
OUTPUT_CSV = DRIVE_BASE / 'stream_b_features.csv'

# ROI: Ventral Striatum / Nucleus Accumbens (MNI coordinates)
# Coordinates from Knutson et al. (2001), commonly used in reward fMRI
VS_SEEDS = [
    (-12, 10, -6),   # Left NAcc
    (12, 10, -6),    # Right NAcc
]
ROI_RADIUS_MM = 6

# Testing mode — set to None for full dataset
MAX_SUBJECTS = 5  # 👈 Change to None for full run

print(f'Drive output: {OUTPUT_CSV}')
print(f'ROI seeds: {VS_SEEDS}, radius={ROI_RADIUS_MM}mm')
print(f'Max subjects: {MAX_SUBJECTS or "ALL"}')

---
## Cell 3: Download & Audit `participants.tsv`

Before touching any imaging data, we audit the phenotypic measures to confirm anhedonia-related scores exist.

In [ ]:
# ============================================================
# Cell 3: Download participants.tsv and audit phenotypes
# ============================================================
import subprocess

# Download just participants.tsv (tiny file, no need for full dataset yet)
PARTICIPANTS_URL = 'https://s3.amazonaws.com/openneuro.org/ds000030/participants.tsv'
PARTICIPANTS_FILE = WORK_DIR / 'participants.tsv'

if not PARTICIPANTS_FILE.exists():
    print('Downloading participants.tsv from OpenNeuro...')
    !wget -q -O {PARTICIPANTS_FILE} {PARTICIPANTS_URL}

# Load and inspect
df_pheno = pd.read_csv(PARTICIPANTS_FILE, sep='\t')
print(f'Total participants: {len(df_pheno)}')
print(f'\nColumns ({len(df_pheno.columns)}):')
print(df_pheno.columns.tolist())

# ---- Identify anhedonia-related measures ---- #
anhedonia_keywords = ['BDI', 'MASQ', 'PHQ', 'BIS', 'BAS', 'TEPS', 'SHAPS', 'anhedon']
relevant_cols = [c for c in df_pheno.columns 
                 if any(kw.lower() in c.lower() for kw in anhedonia_keywords)]

print(f'\n🎯 Anhedonia-related columns found: {relevant_cols}')
if relevant_cols:
    print('\nSummary stats for relevant measures:')
    print(df_pheno[relevant_cols].describe())
else:
    print('\n⚠️ No exact keyword matches. Printing ALL columns for manual inspection:')
    for col in df_pheno.columns:
        print(f'  {col}: {df_pheno[col].dtype} — {df_pheno[col].nunique()} unique values')

---
## Cell 4: Identify Available Tasks & Check Derivatives

ds000030 includes multiple fMRI tasks. We need to identify which task involves **reward processing** (for ventral striatum activation).

In [ ]:
# ============================================================
# Cell 4: Check available fMRI tasks in ds000030
# ============================================================

# ds000030 tasks (from dataset documentation):
# - bart    : Balloon Analogue Risk Task (risk/reward)
# - bht     : Breath Holding Task
# - pamenc  : Paired Associates Memory Encoding
# - pamret  : Paired Associates Memory Retrieval 
# - rest    : Resting State
# - scap    : Spatial Capacity (working memory)
# - stopsignal : Stop-Signal Task (inhibition)
# - taskswitch : Task Switching
#
# For reward/anhedonia: BART is the primary candidate
# (Balloon Analogue Risk Task involves reward-related decision making)

TARGET_TASK = 'bart'  # Reward-related task
SPACE = 'MNI152NLin2009cAsym'  # fMRIPrep standard output space

print(f'Target task: {TARGET_TASK} (Balloon Analogue Risk Task)')
print(f'Target space: {SPACE}')
print(f'\nRationale: BART involves reward-related decision making, which')
print(f'engages the ventral striatum / NAcc — our ROI for anhedonia.')
print(f'\n📋 Next: Download fMRIPrep derivatives for this task.')

---
## Cell 5: Download fMRIPrep Derivatives

Using `openneuro-py` to download preprocessed derivatives. This avoids running fMRIPrep from scratch (~800 CPU-hours).

> ⚠️ **If derivatives are NOT available on OpenNeuro:** you'll need to download raw BOLD data and run fMRIPrep on USC CARC. See the CARC instructions in the project README.

In [ ]:
# ============================================================
# Cell 5: Download or locate fMRIPrep derivatives
# ============================================================
import openneuro

DERIV_DIR = WORK_DIR / 'derivatives' / 'fmriprep'

# Check if derivatives already exist (from a previous run or manual download)
if DERIV_DIR.exists():
    existing = sorted([d.name for d in DERIV_DIR.iterdir() if d.name.startswith('sub-')])
    print(f'Found {len(existing)} existing preprocessed subjects')
else:
    print('No local derivatives found. Attempting OpenNeuro download...')
    print('(This may take a while for the full dataset)\n')
    
    try:
        # Download only the derivatives folder
        openneuro.download(
            dataset='ds000030',
            target_dir=str(WORK_DIR),
            include=['derivatives/fmriprep/'],
            verify_size=False
        )
        print('✅ Derivatives downloaded')
    except Exception as e:
        print(f'⚠️ OpenNeuro download failed: {e}')
        print('\nFallback options:')
        print('  1. Download raw data + run fMRIPrep on CARC')
        print('  2. Use DataLad: datalad install -s https://github.com/OpenNeuroDatasets/ds000030.git')
        print('  3. Manual download from https://openneuro.org/datasets/ds000030')
        print('\nFor now, set DERIV_DIR to your local fMRIPrep output path:')
        print('  DERIV_DIR = Path("/your/path/to/fmriprep")')

---
## Cell 6: ROI Extraction — Ventral Striatum BOLD Features

For each subject with BART task data, extract:
- Mean BOLD signal in bilateral NAcc (6mm sphere)
- BOLD variability (std)
- Left vs Right asymmetry

These features feed into the Fusion analysis alongside Stream A acoustic features.

In [ ]:
# ============================================================
# Cell 6: ROI Extraction Pipeline
# ============================================================
from nilearn.maskers import NiftiSpheresMasker
from nilearn import image
import nibabel as nib
import glob

# Build the spherical ROI masker
masker = NiftiSpheresMasker(
    seeds=VS_SEEDS,
    radius=ROI_RADIUS_MM,
    standardize='zscore_sample',
    detrend=True,
    high_pass=0.01,
    t_r=2.0,  # ds000030 TR
    verbose=0
)

# Find all subjects with fMRIPrep output
if not DERIV_DIR.exists():
    print('❌ No derivatives directory found. Run Cell 5 first.')
else:
    all_subs = sorted([d.name for d in DERIV_DIR.iterdir() 
                       if d.name.startswith('sub-') and d.is_dir()])
    
    if MAX_SUBJECTS:
        all_subs = all_subs[:MAX_SUBJECTS]
    
    print(f'Processing {len(all_subs)} subjects...')
    
    results = []
    errors = []
    
    for sub_id in tqdm(all_subs, desc='ROI extraction'):
        try:
            # Find BART BOLD file in MNI space
            func_dir = DERIV_DIR / sub_id / 'func'
            if not func_dir.exists():
                errors.append((sub_id, 'No func directory'))
                continue
            
            # Pattern: sub-*_task-bart_*space-MNI*_desc-preproc_bold.nii.gz
            bold_files = list(func_dir.glob(
                f'{sub_id}_task-{TARGET_TASK}_*space-{SPACE}*_desc-preproc_bold.nii.gz'
            ))
            
            if not bold_files:
                # Try alternative naming
                bold_files = list(func_dir.glob(
                    f'{sub_id}_task-{TARGET_TASK}*_bold.nii.gz'
                ))
            
            if not bold_files:
                errors.append((sub_id, f'No BART BOLD file found in {func_dir}'))
                continue
            
            bold_path = bold_files[0]
            
            # Extract ROI time series
            signals = masker.fit_transform(str(bold_path))
            # signals shape: (n_timepoints, 2) — [left_NAcc, right_NAcc]
            
            left_ts = signals[:, 0]
            right_ts = signals[:, 1]
            
            results.append({
                'subject_id': sub_id,
                'nacc_left_mean': np.mean(left_ts),
                'nacc_left_std': np.std(left_ts),
                'nacc_right_mean': np.mean(right_ts),
                'nacc_right_std': np.std(right_ts),
                'nacc_bilateral_mean': np.mean(signals),
                'nacc_bilateral_std': np.std(signals),
                'nacc_asymmetry': np.mean(right_ts) - np.mean(left_ts),
                'n_volumes': signals.shape[0],
                'bold_file': bold_path.name
            })
            
        except Exception as e:
            errors.append((sub_id, str(e)))
    
    print(f'\n✅ Successfully processed: {len(results)}/{len(all_subs)}')
    if errors:
        print(f'⚠️ Errors ({len(errors)}):')
        for sub, err in errors[:10]:
            print(f'   {sub}: {err}')

---
## Cell 7: Merge with Phenotype Data & Export CSV

In [ ]:
# ============================================================
# Cell 7: Merge features with phenotypes, export to Drive
# ============================================================

if results:
    df_features = pd.DataFrame(results)
    
    # Merge with phenotype data
    # participants.tsv uses 'participant_id' column (e.g., 'sub-10159')
    df_merged = df_features.merge(
        df_pheno, 
        left_on='subject_id', 
        right_on='participant_id', 
        how='left'
    )
    
    # Save to Drive
    df_merged.to_csv(OUTPUT_CSV, index=False)
    print(f'✅ Saved {len(df_merged)} subjects to {OUTPUT_CSV}')
    print(f'\nFeature columns: {[c for c in df_features.columns if c != "bold_file"]}')
    print(f'\nFirst 5 rows:')
    display(df_features.head())
else:
    print('❌ No results to export. Check Cell 6 for errors.')

---
## Cell 8: Sanity Check Visualizations

In [ ]:
# ============================================================
# Cell 8: Sanity checks — visualize distributions
# ============================================================

if results:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('Stream B Sanity Checks — Ventral Striatum BOLD Features', 
                 fontsize=14, fontweight='bold')
    
    # 1. Bilateral NAcc mean activation
    axes[0, 0].hist(df_features['nacc_bilateral_mean'], bins=20, color='steelblue', edgecolor='white')
    axes[0, 0].set_title('Bilateral NAcc Mean Activation')
    axes[0, 0].set_xlabel('Mean BOLD (z-scored)')
    
    # 2. Bilateral NAcc variability
    axes[0, 1].hist(df_features['nacc_bilateral_std'], bins=20, color='coral', edgecolor='white')
    axes[0, 1].set_title('Bilateral NAcc BOLD Variability')
    axes[0, 1].set_xlabel('Std of BOLD signal')
    
    # 3. Left vs Right asymmetry
    axes[1, 0].hist(df_features['nacc_asymmetry'], bins=20, color='mediumpurple', edgecolor='white')
    axes[1, 0].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[1, 0].set_title('NAcc L-R Asymmetry')
    axes[1, 0].set_xlabel('Right - Left mean')
    
    # 4. Number of volumes per subject
    axes[1, 1].hist(df_features['n_volumes'], bins=20, color='seagreen', edgecolor='white')
    axes[1, 1].set_title('Number of BOLD Volumes')
    axes[1, 1].set_xlabel('Volumes (timepoints)')
    
    plt.tight_layout()
    
    # Save figure to Drive
    fig_path = DRIVE_BASE / 'stream_b_sanity_check.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'📊 Figure saved to {fig_path}')
    plt.show()
else:
    print('No data to visualize.')